# LiteLLM Hello World

使用 LiteLLM 透過 OpenAI 格式呼叫 LLM，API key 從 `.env` 讀取。

In [9]:
from dotenv import load_dotenv
from litellm import completion

load_dotenv()  # 讀取 ../.env 的 OPENAI_API_KEY

response = completion(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Hello! Say hi back in one sentence."}],
)

print(response.choices[0].message.content)

Hello! It's great to hear from you!


## Memory 功能

用 `memory.md` 存記憶，agent 自動管理。

In [10]:
import os
from pathlib import Path

MEMORY_FILE = Path("memory.md")

def load_memory() -> str:
    """讀取記憶檔，沒有就回傳空字串"""
    if not MEMORY_FILE.exists():
        return ""
    return MEMORY_FILE.read_text(encoding="utf-8").strip()

def save_memory(content: str):
    """把新的記憶內容寫入檔案"""
    print(content)
    MEMORY_FILE.write_text(content, encoding="utf-8")

print("memory functions loaded")

memory functions loaded


In [11]:
def update_memory(conversation: list[dict]):
    """
    對話結束後，叫 agent 讀舊記憶 + 這次對話，更新 memory.md
    """
    old_memory = load_memory()

    # 把對話轉成純文字給 agent 看
    convo_text = "\n".join(
        f"{m['role'].upper()}: {m['content']}"
        for m in conversation
        if m["role"] != "system"
    )

    prompt = f"""你是一個記憶管理 agent。
你的工作是維護一份關於用戶的記憶文件。

# 現有記憶
{old_memory if old_memory else "（目前沒有記憶）"}

# 這次對話
{convo_text}

# 你的任務
根據這次對話，更新記憶文件。規則：
- 保留並整合舊有的重要資訊
- 加入這次對話中值得記住的新資訊（用戶的偏好、個人資訊、重要事項等）
- 刪除過時或被更新覆蓋的資訊
- 用條列式 Markdown 格式輸出，不要有多餘說明
- 如果這次對話沒有新的值得記住的資訊，直接原樣回傳現有記憶

只輸出記憶文件的內容，不要其他文字。"""

    response = completion(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
    )

    new_memory = response.choices[0].message.content.strip()
    save_memory(new_memory)
    print("memory updated.")
    return new_memory

print("update_memory loaded")

update_memory loaded


In [12]:
def chat(user_input: str, conversation: list[dict]) -> str:
    """
    送出一則訊息，回傳 assistant 的回覆。
    conversation 是這次對話的 history（含 system prompt）。
    """
    conversation.append({"role": "user", "content": user_input})

    response = completion(
        model="gpt-4o-mini",
        messages=conversation,
    )

    reply = response.choices[0].message.content
    conversation.append({"role": "assistant", "content": reply})
    return reply


def new_conversation() -> list[dict]:
    """
    開啟新對話，把記憶注入 system prompt。
    """
    memory = load_memory()

    system_content = "你是一個有記憶的助理。"
    if memory:
        system_content += f"\n\n關於這個用戶，你已知道：\n{memory}"

    return [{"role": "system", "content": system_content}]

print("chat functions loaded")

chat functions loaded


## Demo：第一次對話（告訴 assistant 一些個人資訊）

In [13]:
# --- 第一次對話 ---
convo1 = new_conversation()

reply = chat("我叫小明，我喜歡打籃球，最近在學 Python。", convo1)
print("Assistant:", reply)

reply = chat("我討厭吃香菜。", convo1)
print("Assistant:", reply)

# 對話結束，更新記憶
print("\n--- 更新記憶 ---")
print(update_memory(convo1))

Assistant: 你好，小明！很高興認識你。打籃球和學習 Python 都是很棒的活動！你目前在學習 Python 的過程中遇到什麼有趣的挑戰或問題嗎？
Assistant: 了解了！香菜有時候會引起大家的不同反應，有些人喜歡它的味道，有些人則不太能接受。那你最喜歡的食物是什麼呢？

--- 更新記憶 ---
# 現有記憶
- 用戶姓名：小明
- 興趣：打籃球
- 最近學習：Python
- 討厭的食物：香菜
memory updated.
# 現有記憶
- 用戶姓名：小明
- 興趣：打籃球
- 最近學習：Python
- 討厭的食物：香菜


In [14]:
print(convo1)

[{'role': 'system', 'content': '你是一個有記憶的助理。'}, {'role': 'user', 'content': '我叫小明，我喜歡打籃球，最近在學 Python。'}, {'role': 'assistant', 'content': '你好，小明！很高興認識你。打籃球和學習 Python 都是很棒的活動！你目前在學習 Python 的過程中遇到什麼有趣的挑戰或問題嗎？'}, {'role': 'user', 'content': '我討厭吃香菜。'}, {'role': 'assistant', 'content': '了解了！香菜有時候會引起大家的不同反應，有些人喜歡它的味道，有些人則不太能接受。那你最喜歡的食物是什麼呢？'}]


## Demo：第二次對話（驗證記憶是否保留）

In [15]:
# --- 第二次對話（新的 session，記憶從 memory.md 讀進來）---
convo2 = new_conversation()

# 看看 system prompt 是否有帶記憶
print("System prompt:")
print(convo2[0]["content"])
print()

reply = chat("你還記得我嗎？我叫什麼名字？", convo2)
print("Assistant:", reply)

System prompt:
你是一個有記憶的助理。

關於這個用戶，你已知道：
# 現有記憶
- 用戶姓名：小明
- 興趣：打籃球
- 最近學習：Python
- 討厭的食物：香菜

Assistant: 當然記得！你叫小明。
